# Assignment 2: Preparation and Testing

In order to perform Assignment 2 in the course, I will use the local PostgreSQL connection and set of queries, checking for the hypotheses I've listed.

In [19]:
from sqlalchemy import create_engine
from sqlalchemy import text
from dotenv import load_dotenv
import os
import pandas as pd
import plotly.express as px

## 1. Create a connection to local database:

In [5]:
load_dotenv()
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")

if not db_name or not db_user:
    raise ValueError("Error: DB_NAME or DB_USER not found in environment variables. Check your .env file!")

engine = create_engine(f"postgresql://{db_user}@localhost/{db_name}")
print("SQLalchemy engine was created and can be used.")

SQLalchemy engine was created and can be used.


## 2. Test a query:

In [16]:
query_test = """SELECT g.YEAR as year, COUNT(DISTINCT t.team_id) AS team_count
FROM team AS t
JOIN athlete_games AS ag ON t.team_id=ag.team_id
JOIN game AS g ON ag.game_id=g.game_id
GROUP BY g.YEAR
ORDER BY g.YEAR;"""

In [10]:
with engine.connect() as connection:
    result = connection.execute(
        text(query_test)
    )
    
    # Iterate through rows
    for row in result:
        print(f"Year: {row[0]}, Athletes: {row[1]}")

Year: 1896, Athletes: 20
Year: 1900, Athletes: 212
Year: 1904, Athletes: 86
Year: 1906, Athletes: 55
Year: 1908, Athletes: 75
Year: 1912, Athletes: 105
Year: 1920, Athletes: 72
Year: 1924, Athletes: 96
Year: 1928, Athletes: 87
Year: 1932, Athletes: 74
Year: 1936, Athletes: 133
Year: 1948, Athletes: 138
Year: 1952, Athletes: 160
Year: 1956, Athletes: 150
Year: 1960, Athletes: 199
Year: 1964, Athletes: 201
Year: 1968, Athletes: 145
Year: 1972, Athletes: 157
Year: 1976, Athletes: 126
Year: 1980, Athletes: 112
Year: 1984, Athletes: 179
Year: 1988, Athletes: 211
Year: 1992, Athletes: 240
Year: 1994, Athletes: 101
Year: 1996, Athletes: 246
Year: 1998, Athletes: 108
Year: 2000, Athletes: 243
Year: 2002, Athletes: 115
Year: 2004, Athletes: 258
Year: 2006, Athletes: 114
Year: 2008, Athletes: 293
Year: 2010, Athletes: 117
Year: 2012, Athletes: 245
Year: 2014, Athletes: 120
Year: 2016, Athletes: 249


In [17]:
df_test = pd.read_sql(text(query_test),con=engine)

In [21]:
fig = px.line(df_test, x='year', y='team_count', title='Number of Olympic Teams (1896-2016)', markers=True)
fig.show()

## 3. Query Development
Now the task is to automate the process: to create a set of queries --> to run them all in one Python script --> to create a visualization for each of them. The number of queries corresponds to the following list of assumptions:

### Assumptions to test
> **Assumption 1**: Female participation rates and medal achievements have experienced a sharp upward inflection point in the latter half of the 20th century, particularly following institutional policy changes in international sports federations.<br>

Possible query:
```SQL
-- ratio of women over years:
SELECT
    g.YEAR AS year,
    --COUNT(DISTINCT CASE WHEN a.sex = 'F' THEN a.athlete_id END) AS women_count,
    --COUNT(DISTINCT a.athlete_id) AS total_participants,
    (COUNT(DISTINCT CASE WHEN a.sex = 'F' THEN a.athlete_id END) * 100.0) / COUNT(DISTINCT a.athlete_id) AS share_of_women
FROM athlete AS a
JOIN athlete_games AS ag ON a.athlete_id = ag.athlete_id
JOIN game AS g ON ag.game_id = g.game_id
GROUP BY g.YEAR
ORDER BY g.YEAR;
```
**Interesting Observation**
Yes, the share of female athletes who participated in the OG grows but it started a bit ealier than 1980s: already in 1972 there were almost 16% of women with 20.7% in 1976. The increase in the 80s was pretty steady: 1980 -- 21.5%, 1984 -- 22.9%, 1988 -- 25.5% (about 2% every round of the OG). Unfortunately, but 2016 women are still minority: just 44.9% of athletes participating in the OG.

--- 

> **Assumption 2**: Certain countries maintain a persistent specialization advantage in specific event categories (e.g., endurance vs. strength sports) that remains stable across decades regardless of geopolitical shifts.<br>
```SQL 
-- meadals per country over years
SELECT
	r.region AS country,
	g.YEAR AS YEAR,
	COUNT(ra.medal_type) AS medals_count
FROM region AS r
JOIN team AS t ON r.noc_id=t.noc_id
JOIN athlete_games AS ag ON t.team_id=ag.team_id
JOIN result_award AS ra on ag.participation_id=ra.participation_id
JOIN game AS g ON ag.game_id=g.game_id
GROUP BY r.region, g.YEAR
ORDER BY g.YEAR;

-- check if there are countries with max medals over all the years:
-- medals per country per sport iver years
WITH medals_over_years AS (
SELECT
	r.region AS country,
	g.YEAR AS YEAR,
	e.sport,
	COUNT(ra.medal_type) AS medals_count
FROM region AS r
JOIN team AS t ON r.noc_id=t.noc_id
JOIN athlete_games AS ag ON t.team_id=ag.team_id
JOIN result_award AS ra on ag.participation_id=ra.participation_id
JOIN game AS g ON ag.game_id=g.game_id
JOIN EVENT AS e ON ag.event_id = e.event_id
GROUP BY g.YEAR, r.region, e.sport
ORDER BY g.YEAR, r.region, e.sport
)
-- medals one in particular sport over many years, sorted
SELECT country, sport,COUNT(year)
FROM medals_over_years
GROUP BY sport, country
HAVING count(year) >= 10
ORDER BY sport, COUNT(YEAR) DESC;
``` 

**Interesting Observations**:
* Alipine Skiing is dominated by Austria, Switzerland, the USA, France and Germany;
* Archery -- by the USA and South Korea;
* Athletics -- by UK, the USA, France, Germany, Australia, and Finland;
* Basketball -- by the USA and Russia;
* Biathlon -- by Russia and Germany;
* Bobsleigh -- by Switzerland and Germany;
* Boxing -- by the USA and the UK;
* Judo -- by Germany and Japan;
* Shooting -- by the USA and Sweden;
* Ice Hockey -- by Canada, the USA, and Russia,
* etc

--- 

> **Assumption 2a** (added): A small elite group of countries (e.g., USA, Soviet Union/Russia, Germany, UK) captures a disproportionately high percentage (e.g., over 50%) of all medals awarded historically.<br>
```SQL
-- medals per country per year (optionally -- per sport discipline)
WITH medals_over_years AS (
    SELECT
        r.region AS country,
        g.YEAR AS YEAR,
        e.sport,
        COUNT(ra.medal_type) AS medals_count
    FROM region AS r
    JOIN team AS t ON r.noc_id = t.noc_id
    JOIN athlete_games AS ag ON t.team_id = ag.team_id
    JOIN result_award AS ra ON ag.participation_id = ra.participation_id
    JOIN game AS g ON ag.game_id = g.game_id
    JOIN EVENT AS e ON ag.event_id = e.event_id
    GROUP BY g.YEAR, r.region, e.sport
),
-- medals per country over years
country_totals AS (
    SELECT 
        country,
        SUM(medals_count) AS total_medals
    FROM medals_over_years
    GROUP BY country
),
-- ranking from 0 to 1
ranked_countries AS (
    SELECT 
        country,
        total_medals,
        PERCENT_RANK() OVER (ORDER BY total_medals DESC) AS pct_rank
    FROM country_totals
),
-- tiers (can be adjusted in naming)
tiers AS 
(SELECT 
    country,
    total_medals,
    CASE 
        WHEN pct_rank <= 0.10 THEN 'Elite (Top 10%)'
        WHEN pct_rank <= 0.25 THEN 'High Performing (Top 25%)'
        ELSE 'Standard'
    END AS tier
FROM ranked_countries
)
-- checking the share of medals per Top 10% Tier vs all the other tiers
SELECT
	SUM(total_medals) AS total_medals,
    SUM(total_medals) FILTER (WHERE tier = 'Elite (Top 10%)') AS total_elite_medals,
    SUM(total_medals) FILTER (WHERE tier != 'Elite (Top 10%)') AS total_not_elite_medals,
    
    ROUND(100.0* SUM(total_medals) FILTER (WHERE tier = 'Elite (Top 10%)') / SUM(total_medals), 2) AS elite_share,
    ROUND(100.0* SUM(total_medals) FILTER (WHERE tier != 'Elite (Top 10%)') / SUM(total_medals), 2) AS not_elite_share
FROM tiers;
```
**Observation**:<br>
Here I'm checking if ~20% participating countries are taking about ~80% of the medals (aka Pareto principle): so I group countries dynamically, until there "cumulative percent" will reach 80% of the total medals bank. There are other approaches to decide, if a country belongs to "elite" group (e.g., the biggest teams or the biggest budgets for sports etc).<br>
* Examples of elite countries: the USA (10 870 medals), Russia (8 448 medalsor 2), Germany (7 244 medals), etc.
* Examples of not_elite countries: Nepal, Macedonia, Monaco, Iraq and many other countries with 1 medal; Malta, Niger, Zambia (and many other countries) with 2 medals, ... Bulgaria (with 490 medals), Austria (928 medals), Czeck Republic (1 240 medals).
By using this approach, elite countries take ~74% of the total medals bank, when ~26% goes to "not-elite" (below 10% top tier) countries.

--- 

> **Assumption 3** (re-formulated): While the overall average age of all Olympians hovers consistently around 24–26 years old, specific sports like Gymnastics and Figure Skating have drastically younger average ages (teens) compared to Equestrian or Sailing (late 30s+).<br>
```SQL
SELECT 
	e.sport AS sport_discipline,
	AVG(ag.age) AS average_age
FROM athlete_games AS ag
JOIN EVENT AS e ON ag.event_id = e.event_id
GROUP BY e.sport
ORDER BY AVG(ag.age) DESC;
```
**Comments and Observations**
I calculated average age of participation and not taking into account individual unique athletes, entering the OG.
* If we look at the average participation age across all the games and sports, we will have **25.48 years**:
```SQL
SELECT AVG(age) FROM athlete_games;
```
* If we include biological sex into this analysis, average age for women will be **23.7** and for men **26.2**:
```SQL
SELECT a.sex, AVG(ag.age) 
FROM athlete_games AS ag
JOIN athlete AS a ON ag.athlete_id = a.athlete_id
GROUP BY a.sex;
```
* I also could check how average age was changing over the years:
```SQL
SELECT 
	g.YEAR,
	AVG(ag.age) 
FROM athlete_games AS ag
JOIN game AS g ON ag.game_id = g.game_id
GROUP BY g.YEAR
ORDER BY g.year;
``` 
**BETTER ADD A VISUALIZATION HERE!!**

* And now some descriptive stats by sport disciplines:
```SQL
SELECT 
	e.sport AS sport_discipline,
	AVG(ag.age) AS average_age
FROM athlete_games AS ag
JOIN EVENT AS e ON ag.event_id = e.event_id
GROUP BY e.sport
ORDER BY AVG(ag.age) DESC;
```
> There are some sports with relatively high average participation ages (roque -- 53, art competitions -- 44.7, alpinism -- 38.8, polo -- 34.9, equestrianism -- 34.4, croquet -- 33.7 etc.)
> There are several sport with relatively low average participation ages (phytmic gymnastics -- 18.7, swimming -- 20.5, figure skating -- 22.2, synchronized swimming -- 22.4, diving -- 22.5, gymnastics -- 22.8 etc.)
> So, yes, average ages are sport specific. But it would be interesting to see how many "veteran" athletes exist in each sport and how this can affect the averages + check the ranges as well to have a better picture (boxplot would be great).


--- 

> **Assumption 4**: Across physical sports (like basketball, swimming, and rowing), the average height and weight of athletes have grown steadily over the decades due to professionalization, training standards, and nutrition.<br>

> **Assumption 4a**: HEIGHT!
```SQL
-- average height of participation over years:
SELECT COALESCE(AVG(height)) FROM athlete_games;

---- by gender:
SELECT a.sex, COALESCE(AVG(ag.height)) 
FROM athlete_games AS ag
JOIN athlete AS a ON ag.athlete_id = a.athlete_id
GROUP BY a.sex;

-- average height of participation per year:
SELECT 
	g.YEAR,
	COALESCE(AVG(ag.height))
FROM athlete_games AS ag
JOIN game AS g ON ag.game_id = g.game_id
GROUP BY g.YEAR
ORDER BY g.year;


-- average height of participation over years per sport:
SELECT 
	e.sport AS sport_discipline,
	COALESCE(AVG(ag.height)) AS average_height
FROM athlete_games AS ag
JOIN EVENT AS e ON ag.event_id = e.event_id
WHERE ag.height IS NOT NULL
GROUP BY e.sport
ORDER BY COALESCE(AVG(ag.height)) DESC;
```

> **Assumption 4b**: WEIGHT!
```SQL
-- average weight of participation over years:
SELECT COALESCE(AVG(weight)) FROM athlete_games;

---- by gender:
SELECT a.sex, COALESCE(AVG(ag.weight)) 
FROM athlete_games AS ag
JOIN athlete AS a ON ag.athlete_id = a.athlete_id
GROUP BY a.sex;

-- average weight of participation per year:
SELECT 
	g.YEAR,
	COALESCE(AVG(ag.weight))
FROM athlete_games AS ag
JOIN game AS g ON ag.game_id = g.game_id
GROUP BY g.YEAR
ORDER BY g.year;


-- average weight of participation over years per sport:
SELECT 
	e.sport AS sport_discipline,
	COALESCE(AVG(ag.weight)) AS average_weight
FROM athlete_games AS ag
JOIN EVENT AS e ON ag.event_id = e.event_id
WHERE ag.weight IS NOT NULL
GROUP BY e.sport
ORDER BY COALESCE(AVG(ag.weight)) DESC;
```

--- 

> **Assumption 5**: The total number of unique countries/regions (region via noc_regions) participating in the Olympics has grown significantly, reflecting geopolitical shifts (such as post-WWII and post-Soviet independence waves).<br>
```SQL
-- total number of unique countries ever participating in the OG:
SELECT COUNT(DISTINCT team_id) FROM athlete_games;

-- number of countries-participants per year:
SELECT
	g.YEAR,
	COUNT(DISTINCT ag.team_id) AS participating_countries
FROM athlete_games AS ag
JOIN game AS g ON ag.game_id = g.game_id
GROUP BY g.YEAR
ORDER BY g.YEAR;
```
* Need a visualization and deeper historical analysis.

--- 

> **Assumprion 6**: The total number of distinct events (event) and sports (sport) has expanded aggressively, particularly in the Summer Games, through the introduction of new disciplines and weight classes.<br>
```SQL
-- total amount of events in 2016 (latest entry):
SELECT
	COUNT(DISTINCT e.event_id)
FROM game AS g
JOIN athlete_games AS AG ON g.game_id = ag.game_id
JOIN EVENT AS e ON ag.event_id = e.event_id
WHERE g.YEAR=2016;

-- over years:
SELECT
	g.YEAR,
	g.season,
	COUNT(DISTINCT e.event_id) AS total_unique_events
FROM game AS g
JOIN athlete_games AS AG ON g.game_id = ag.game_id
JOIN EVENT AS e ON ag.event_id = e.event_id
GROUP BY g.YEAR, g.season
ORDER BY g.year;

-- verification:
SELECT
	g.game_id,
	YEAR,
	season,
	COUNT(DISTINCT ag.event_id) AS total_unique_events
FROM game AS g
JOIN athlete_games AS ag ON g.game_id = ag.game_id
WHERE g.YEAR = 1920 AND g.season='Summer'
GROUP BY g.game_id, g.YEAR, g.season;
```

---

> **Assumption 7**: Summer Games account for the vast majority of total rows, unique athletes, and participating nations compared to Winter Games, which remain structurally smaller in scale.<br>

See above. This is true. But we can re-verify:

> **Assumption 8**: Host nations experience a temporary, statistically significant surge in medal production (especially gold medals) during the specific year they host the Games.<br>

> **Assumption 9**: The vast majority of Olympic athletes participate in only a single Olympic Games, while multi-game Olympians (competing in 3+ games) represent a tiny elite minority.<br>